# Extração de características baseadas em HOG e LBP & Aplicação do Método PCA

## Donwload do dataset

In [1]:
# Dataset: 27,shiba_inu,saint_bernard,Egyptian_Mau,Russian_Blue

# Importando as libs
from google.colab import files
import os, io, time

# Criando o diretório inicial:
os.chdir('/content/')
try:
  os.mkdir('Imagens', )
except:
  print('A pasta já existe.')

os.chdir('./Imagens')
os.listdir()
uploaded_images = files.upload()

Saving Egyptian_Mau_1.jpg to Egyptian_Mau_1.jpg
Saving Egyptian_Mau_2.jpg to Egyptian_Mau_2.jpg
Saving Egyptian_Mau_3.jpg to Egyptian_Mau_3.jpg
Saving Egyptian_Mau_4.jpg to Egyptian_Mau_4.jpg
Saving Egyptian_Mau_5.jpg to Egyptian_Mau_5.jpg
Saving Egyptian_Mau_6.jpg to Egyptian_Mau_6.jpg
Saving Egyptian_Mau_7.jpg to Egyptian_Mau_7.jpg
Saving Egyptian_Mau_8.jpg to Egyptian_Mau_8.jpg
Saving Egyptian_Mau_19.jpg to Egyptian_Mau_19.jpg
Saving Egyptian_Mau_20.jpg to Egyptian_Mau_20.jpg
Saving Egyptian_Mau_21.jpg to Egyptian_Mau_21.jpg
Saving Egyptian_Mau_33.jpg to Egyptian_Mau_33.jpg
Saving Egyptian_Mau_34.jpg to Egyptian_Mau_34.jpg
Saving Egyptian_Mau_35.jpg to Egyptian_Mau_35.jpg
Saving Egyptian_Mau_47.jpg to Egyptian_Mau_47.jpg
Saving Egyptian_Mau_48.jpg to Egyptian_Mau_48.jpg
Saving Egyptian_Mau_49.jpg to Egyptian_Mau_49.jpg
Saving Egyptian_Mau_60.jpg to Egyptian_Mau_60.jpg
Saving Egyptian_Mau_61.jpg to Egyptian_Mau_61.jpg
Saving Egyptian_Mau_62.jpg to Egyptian_Mau_62.jpg
Saving Egyptian_

## Redução das imagens

In [21]:
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import numpy as np
import warnings
warnings.filterwarnings('ignore')

images_256 = {}
images_128 = {}

for filename in uploaded_images.keys():
    image = imread(filename)

    # Se tiver 4 canais (RGBA), remover o canal alfa
    if image.ndim == 3 and image.shape[2] == 4:
        image = image[:, :, :3]

    # Se for escala de cinza (2D), converter para RGB
    if image.ndim == 2:
        image = np.stack([image] * 3, axis=-1)

    # Reduzir
    image_256 = resize(image, (256, 256, 3))
    image_128 = resize(image, (128, 128, 3))

    # Garantir que ficou com 3 dimensões
    if image_256.ndim == 4:
        image_256 = image_256[:, :, :, 0]
    if image_128.ndim == 4:
        image_128 = image_128[:, :, :, 0]

    images_256[filename] = image_256
    images_128[filename] = image_128

# Verificar as problemáticas
for filename in ['Egyptian_Mau_139.jpg', 'Egyptian_Mau_167.jpg', 'Egyptian_Mau_191.jpg']:
    print(f'{filename} - shape: {images_256[filename].shape}')

# Verificar se todas estão OK
problemas = [f for f in uploaded_images.keys() if images_256[f].shape != (256, 256, 3)]
if len(problemas) == 0:
    print(f'\nTodas as {len(images_256)} imagens OK com shape (256, 256, 3)')
else:
    print(f'\nProblemas em: {problemas}')


Egyptian_Mau_139.jpg - shape: (256, 256, 3)
Egyptian_Mau_167.jpg - shape: (256, 256, 3)
Egyptian_Mau_191.jpg - shape: (256, 256, 3)

Todas as 656 imagens OK com shape (256, 256, 3)


## Classificação das raças: dog e cat

In [22]:
classe = ['dog' if 'shiba_inu' in file
          or 'saint_bernard' in filename
          else 'cat' if 'Egyptian_Mau' in filename
          or 'Russian_Blue' in filename
          else file for file in uploaded_images.keys()]

print(classe)

['cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat'

## Aplicar HOG (6 bases)

In [24]:
# Verificação

for filename in uploaded_images.keys():
    image = images_256[filename]
    print(f'{filename} - shape: {image.shape} - ndim: {image.ndim} - dtype: {image.dtype}')

Egyptian_Mau_1.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_2.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_3.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_4.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_5.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_6.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_7.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_8.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_19.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_20.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_21.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_33.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_34.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_35.jpg - shape: (256, 256, 3) - ndim: 3 - dtype: float64
Egyptian_Mau_47.jpg - shape:

In [26]:
from skimage.feature import hog
import pandas as pd
import numpy as np

hog_configs = [
    (images_256, '256', 8),
    (images_256, '256', 16),
    (images_256, '256', 32),
    (images_128, '128', 8),
    (images_128, '128', 16),
    (images_128, '128', 32),
]

classe = ['dog' if 'shiba_inu' in file
          or 'saint_bernard' in file
          else 'cat' if 'Egyptian_Mau' in file
          or 'Russian_Blue' in file
          else file for file in uploaded_images.keys()]

for images_dict, res_label, ppc in hog_configs:
    hog_features = []

    for filename in uploaded_images.keys():
        image_resized = images_dict[filename]
        fd, hog_image = hog(image_resized, orientations=9,
                            pixels_per_cell=(ppc, ppc),
                            cells_per_block=(2, 2),
                            visualize=True,
                            channel_axis=-1)
        hog_features.append(fd)

    data = {"hog_features": hog_features, "classe": classe}
    df = pd.DataFrame(data)

    df2 = pd.DataFrame(df['hog_features'].tolist())
    df2.columns = df2.columns.map(lambda x: f'hog_feature_{x+1}')
    df = pd.concat([df.drop('hog_features', axis=1), df2], axis=1)
    df['classe'] = df.pop('classe')

    csv_name = f'HOG_{res_label}_{ppc}x{ppc}.csv'
    df.to_csv(csv_name, sep=',', index=False)
    print(f'{csv_name} - Atributos: {df.shape[1]-1}')

HOG_256_8x8.csv - Atributos: 34596
HOG_256_16x16.csv - Atributos: 8100
HOG_256_32x32.csv - Atributos: 1764
HOG_128_8x8.csv - Atributos: 8100
HOG_128_16x16.csv - Atributos: 1764
HOG_128_32x32.csv - Atributos: 324


## LBP

In [27]:
from skimage.feature import local_binary_pattern
import numpy as np

lbp_configs = [3, 6, 12]

for radius in lbp_configs:
    lbp_features = []
    n_points = 8 * radius

    for filename in uploaded_images.keys():
        image_resized = images_256[filename]
        gray = rgb2gray(image_resized)

        lbp = local_binary_pattern(gray, n_points, radius, method="uniform")

        (hist, _) = np.histogram(lbp.ravel(),
                                  bins=np.arange(0, n_points + 3),
                                  range=(0, n_points + 2))
        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-7)
        lbp_features.append(hist)

    data = {"lbp_features": lbp_features, "classe": classe}
    df = pd.DataFrame(data)

    df2 = pd.DataFrame(df['lbp_features'].tolist())
    df2.columns = df2.columns.map(lambda x: f'lbp_feature_{x+1}')
    df = pd.concat([df.drop('lbp_features', axis=1), df2], axis=1)
    df['classe'] = df.pop('classe')

    csv_name = f'LBP_256_{radius}R.csv'
    df.to_csv(csv_name, sep=',', index=False)
    print(f'{csv_name} - Atributos: {df.shape[1]-1}')

LBP_256_3R.csv - Atributos: 26
LBP_256_6R.csv - Atributos: 50
LBP_256_12R.csv - Atributos: 98


## PCA 75 e 90%

In [28]:
from sklearn.decomposition import PCA

hog_files = [
    'HOG_256_8x8.csv',
    'HOG_256_16x16.csv',
    'HOG_256_32x32.csv',
    'HOG_128_8x8.csv',
    'HOG_128_16x16.csv',
]

for hog_file in hog_files:
    df = pd.read_csv(hog_file)

    X = df.drop('classe', axis=1)
    y = df['classe']

    for var in [0.75, 0.90]:
        pca = PCA(n_components=var)
        X_pca = pca.fit_transform(X)

        df_pca = pd.DataFrame(X_pca)
        df_pca.columns = [f'pca_feature_{i+1}' for i in range(df_pca.shape[1])]
        df_pca['classe'] = y.values

        pct = str(int(var * 100)).zfill(3)
        hog_label = hog_file.replace('HOG_', '').replace('.csv', '')
        csv_name = f'PCA_{pct}_HOG_{hog_label}.csv'

        df_pca.to_csv(csv_name, sep=',', index=False)
        print(f'{csv_name} - Atributos: {df_pca.shape[1]-1}')

PCA_075_HOG_256_8x8.csv - Atributos: 320
PCA_090_HOG_256_8x8.csv - Atributos: 487
PCA_075_HOG_256_16x16.csv - Atributos: 199
PCA_090_HOG_256_16x16.csv - Atributos: 363
PCA_075_HOG_256_32x32.csv - Atributos: 86
PCA_090_HOG_256_32x32.csv - Atributos: 175
PCA_075_HOG_128_8x8.csv - Atributos: 240
PCA_090_HOG_128_8x8.csv - Atributos: 408
PCA_075_HOG_128_16x16.csv - Atributos: 105
PCA_090_HOG_128_16x16.csv - Atributos: 210
